# 07·03 — Instruction Tuning: Teaching Models to Follow Directions

> **Module:** 07 – Fine-Tuning  
> **Notebook:** 3 of 4  
> **Prerequisites:** `01_lora_finetuning.ipynb`, `02_qlora.ipynb`  
> **Objective:** Understand the full lifecycle of instruction tuning — from data format decisions to loss masking to evaluating whether the model actually improved.

---

## 🎯 Learning Objectives

1. Understand what instruction tuning does mechanically vs pretraining
2. Master the three major chat template formats and why they differ
3. Implement **prompt masking** correctly — the most common implementation mistake
4. Analyse training dynamics: what healthy vs unhealthy loss curves look like
5. Evaluate instruction-following quality systematically
6. Understand multi-turn fine-tuning and its unique challenges

---

## 1. Pretraining vs Instruction Tuning: What Changes?

### Pretraining
The model learns language structure by predicting the next token across trillions of tokens of raw text. It learns *everything* — facts, reasoning patterns, code syntax, writing styles. But the resulting model is a **completion engine**: given any prefix, it continues plausibly.

```
Input:  "The capital of France is"
Output: "Paris. France is a country in Western Europe..."
                             ↑ just continues plausibly
```

### Instruction Tuning (SFT — Supervised Fine-Tuning)
A small additional training phase on **structured (prompt, response) pairs** that teaches the model a *behaviour*: respond helpfully when given an instruction.

```
Input:  "### Instruction:\nWhat is the capital of France?\n\n### Response:\n"
Output: "The capital of France is Paris."
                             ↑ answers the question directly
```

### The Key Mechanical Difference: Loss Masking

```
PRETRAINING:   compute loss on EVERY token

   Input:  [The] [cat] [sat] [on] [the] [mat]
   Loss:     ✓    ✓    ✓    ✓    ✓    ✓

INSTRUCTION TUNING: compute loss ONLY on response tokens

   Input:  [###] [Inst] [:] [What] ... [###] [Resp] [:] [Paris] [.]
   Loss:     ✗    ✗    ✗    ✗    ✗    ✗    ✗    ✗    ✓      ✓
                                                              ↑ only these
```

Why mask the prompt? Because we don't want the model to learn to predict the user's instruction — we want it to learn good *responses* given instructions. Training on prompt tokens:
1. Wastes gradient signal on tokens the model won't need to generate
2. Can cause the model to become good at predicting instructions rather than answering them
3. May degrade the model's instruction-following if the instructions are low-quality

---

## 2. Chat Template Formats

Different model families use incompatible template formats. Using the wrong template at inference time will badly degrade performance.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install transformers peft datasets trl accelerate matplotlib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
import warnings, json, re
warnings.filterwarnings('ignore')

print("Setup complete.")

In [ ]:
# ── The major chat template formats ───────────────────────────────────────

class ChatTemplates:
    """
    Reference implementations of the four dominant chat template formats.
    
    Using the wrong template at inference is one of the most common 
    mistakes when deploying fine-tuned models — it can drop performance
    by 20-30% on instruction-following benchmarks.
    """

    @staticmethod
    def alpaca(instruction: str, input_: str = "", response: str = "") -> str:
        """
        Alpaca format (Stanford, 2023). Original instruction-tuning template.
        Used by: Alpaca, Vicuna (v1), many early open models.
        """
        if input_:
            prompt = (
                "Below is an instruction that describes a task, paired with an input "
                "that provides further context. Write a response that appropriately "
                "completes the request.\n\n"
                f"### Instruction:\n{instruction}\n\n"
                f"### Input:\n{input_}\n\n"
                "### Response:\n"
            )
        else:
            prompt = (
                "Below is an instruction that describes a task. Write a response that "
                "appropriately completes the request.\n\n"
                f"### Instruction:\n{instruction}\n\n"
                "### Response:\n"
            )
        return prompt + response

    @staticmethod
    def chatml(messages: List[Dict], add_generation_prompt: bool = True) -> str:
        """
        ChatML format (OpenAI-derived). Now the most widely adopted.
        Used by: Mistral, Mixtral, Qwen, phi-3, many 2024 models.
        Special tokens: <|im_start|> and <|im_end|>
        """
        result = ""
        for msg in messages:
            result += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
        if add_generation_prompt:
            result += "<|im_start|>assistant\n"
        return result

    @staticmethod
    def llama2(messages: List[Dict], system: str = "", add_generation_prompt: bool = True) -> str:
        """
        LLaMA-2 chat format. Baroque but widely used.
        Used by: LLaMA-2-chat, CodeLLaMA-instruct.
        Note: LLaMA-3 uses a completely different format (see llama3 method).
        """
        B_INST, E_INST = "[INST]", "[/INST]"
        B_SYS,  E_SYS  = "<<SYS>>\n", "\n<</SYS>>\n\n"
        BOS, EOS = "<s>", "</s>"

        result = ""
        for i, msg in enumerate(messages):
            if msg["role"] == "system":
                continue  # handled below
            if msg["role"] == "user":
                content = msg["content"]
                if i == 0 and system:
                    content = B_SYS + system + E_SYS + content
                result += f"{BOS}{B_INST} {content.strip()} {E_INST}"
            elif msg["role"] == "assistant":
                result += f" {msg['content'].strip()} {EOS}"
        if add_generation_prompt:
            result = result.rstrip(EOS)  # remove trailing EOS if we're generating
        return result

    @staticmethod
    def llama3(messages: List[Dict], add_generation_prompt: bool = True) -> str:
        """
        LLaMA-3 format. Cleaner than LLaMA-2, uses special token IDs.
        Used by: LLaMA-3, LLaMA-3.1, LLaMA-3.2.
        """
        result = "<|begin_of_text|>"
        for msg in messages:
            result += (
                f"<|start_header_id|>{msg['role']}<|end_header_id|>\n\n"
                f"{msg['content'].strip()}<|eot_id|>\n"
            )
        if add_generation_prompt:
            result += "<|start_header_id|>assistant<|end_header_id|>\n\n"
        return result


# ── Demo: same conversation, four formats ─────────────────────────────────
messages = [
    {"role": "system",    "content": "You are a concise technical assistant."},
    {"role": "user",      "content": "What is gradient descent?"},
    {"role": "assistant", "content": "Gradient descent iteratively moves parameters in the direction of steepest loss reduction."},
    {"role": "user",      "content": "What's the learning rate?"},
]

t = ChatTemplates()

print("=" * 60)
print("ALPACA FORMAT")
print("=" * 60)
print(t.alpaca("What is gradient descent?"))

print("\n" + "=" * 60)
print("CHATML FORMAT")
print("=" * 60)
print(t.chatml(messages))

print("\n" + "=" * 60)
print("LLAMA-2 FORMAT")
print("=" * 60)
print(t.llama2(messages, system="You are a concise technical assistant."))

print("\n" + "=" * 60)
print("LLAMA-3 FORMAT")
print("=" * 60)
print(t.llama3(messages))

In [ ]:
# HuggingFace tokenizers now have built-in apply_chat_template
# Always prefer this over hand-rolling templates!

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# The proper way in production — uses the model's own template
hf_messages = [
    {"role": "user",      "content": "What is gradient descent?"},
    {"role": "assistant", "content": "An optimisation algorithm."},
    {"role": "user",      "content": "Explain the learning rate."},
]

# GPT-2 doesn't have a chat template, so we define a simple one for demo
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '### ' + message['role'].upper() + ':\n' + message['content'] + '\n\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '### ASSISTANT:\n' }}{% endif %}"
)

formatted = tokenizer.apply_chat_template(
    hf_messages,
    tokenize=False,
    add_generation_prompt=True
)
print("Formatted via tokenizer.apply_chat_template:")
print(formatted)

print("\n⚠️  CRITICAL: At inference time, ALWAYS use the same template used during training.")
print("   Template mismatch is one of the most common and hardest-to-debug deployment errors.")

## 3. Prompt Masking: The Most Important Implementation Detail

Getting this wrong is the **#1 most common instruction-tuning implementation bug**.

In [ ]:
def visualize_loss_mask(tokenizer, text: str, response_template: str):
    """
    Visualise which tokens are masked (prompt) vs active (response) in the loss.
    """
    tokens = tokenizer.encode(text)
    token_strings = [tokenizer.decode([t]) for t in tokens]

    # Find where the response begins
    response_ids = tokenizer.encode(response_template, add_special_tokens=False)
    response_start = None
    for i in range(len(tokens) - len(response_ids) + 1):
        if tokens[i:i+len(response_ids)] == response_ids:
            response_start = i + len(response_ids)
            break

    mask = [False] * len(tokens)
    if response_start is not None:
        for i in range(response_start, len(tokens)):
            mask[i] = True

    # Visualise
    fig, ax = plt.subplots(figsize=(max(14, len(tokens) * 0.6), 3))
    ax.set_xlim(-0.5, len(tokens) - 0.5)
    ax.set_ylim(-0.5, 1.5)

    for i, (tok, active) in enumerate(zip(token_strings, mask)):
        color = '#2ca02c' if active else '#d62728'
        alpha = 0.8
        rect = plt.Rectangle((i - 0.45, 0.1), 0.9, 0.8,
                               facecolor=color, alpha=alpha, edgecolor='white', linewidth=1.5)
        ax.add_patch(rect)
        display = tok.replace('Ġ', ' ').replace('Ċ', '↵').strip() or '?'
        ax.text(i, 0.5, display[:8], ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
        ax.text(i, 1.2, str(tokens[i]), ha='center', va='center', fontsize=6, color='gray')

    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_yticks([])
    ax.set_xticks([])

    prompt_patch = mpatches.Patch(color='#d62728', label='Prompt (masked — no gradient)')
    resp_patch   = mpatches.Patch(color='#2ca02c', label='Response (active — gradient flows)')
    ax.legend(handles=[prompt_patch, resp_patch], loc='upper right', fontsize=9)

    active_count = sum(mask)
    total_count = len(tokens)
    ax.set_title(
        f'Loss Mask Visualization  |  '
        f'{active_count}/{total_count} tokens active ({100*active_count/total_count:.0f}%)',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('../figures/07_loss_mask.png', dpi=150, bbox_inches='tight')
    plt.show()
    return mask


import matplotlib.patches as mpatches

example_text = (
    "### Instruction:\nExplain gradient descent in one sentence.\n\n"
    "### Response:\nGradient descent minimises a loss by iteratively stepping in the direction of steepest descent."
)

mask = visualize_loss_mask(tokenizer, example_text, response_template="### Response:\n")
print(f"\nTokens in prompt (no gradient): {sum(not m for m in mask)}")
print(f"Tokens in response (gradient):  {sum(mask)}")
print(f"Loss efficiency:                {100*sum(mask)/len(mask):.0f}% of tokens contribute to learning")

In [ ]:
# ── The correct way: DataCollatorForCompletionOnlyLM ───────────────────────
# TRL's data collator handles masking automatically — use this in production.

RESPONSE_TEMPLATE = "### Response:\n"
INSTRUCTION_TEMPLATE = "### Instruction:\n"

collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

def format_instruction(example: Dict) -> str:
    """Format a single example for instruction tuning."""
    return (
        f"{INSTRUCTION_TEMPLATE}{example['instruction']}\n\n"
        f"{RESPONSE_TEMPLATE}{example['response']}"
    )

# Test the collator produces correct masks
test_example = {
    "instruction": "What is a transformer?",
    "response": "A transformer is a neural network architecture using self-attention to process sequences."
}
formatted = format_instruction(test_example)
tokenized = tokenizer(formatted, return_tensors='pt')

# Manually replicate what the collator does
input_ids = tokenized['input_ids'][0]
labels = input_ids.clone()

# Find response start and mask everything before it
response_token_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
response_start_idx = None
for i in range(len(input_ids) - len(response_token_ids) + 1):
    if input_ids[i:i+len(response_token_ids)].tolist() == response_token_ids:
        response_start_idx = i + len(response_token_ids)
        break

if response_start_idx is not None:
    labels[:response_start_idx] = -100  # -100 is PyTorch's ignore index

n_active = (labels != -100).sum().item()
n_total = len(labels)
print(f"Correctly masked: {n_total - n_active} prompt tokens set to -100")
print(f"Active response tokens: {n_active}")
print()
print("The DataCollatorForCompletionOnlyLM does this automatically for every batch.")
print("Always verify your masking is correct before starting a long training run!")

In [ ]:
# ── Bug demonstration: training WITHOUT masking ────────────────────────────
# This is what goes wrong if you skip DataCollatorForCompletionOnlyLM

print("Comparing loss computation: Masked vs Unmasked\n")

tokenizer2 = AutoTokenizer.from_pretrained("gpt2")
tokenizer2.pad_token = tokenizer2.eos_token
model_demo = AutoModelForCausalLM.from_pretrained("gpt2")
model_demo.eval()

prompt = "### Instruction:\nWhat year was Python created?\n\n### Response:\n"
response = "Python was created in 1991 by Guido van Rossum."
full_text = prompt + response

inputs = tokenizer2(full_text, return_tensors='pt')
input_ids = inputs['input_ids']

with torch.no_grad():
    # Case 1: No masking (wrong — trains on prompt tokens too)
    outputs_no_mask = model_demo(input_ids, labels=input_ids)
    loss_no_mask = outputs_no_mask.loss.item()

    # Case 2: With masking (correct)
    labels_masked = input_ids.clone()
    prompt_len = len(tokenizer2.encode(prompt))
    labels_masked[0, :prompt_len] = -100
    outputs_masked = model_demo(input_ids, labels=labels_masked)
    loss_masked = outputs_masked.loss.item()

    # Case 3: Verbose prompt, short response — extreme masking effect
    long_prompt = prompt * 5   # artificially long prompt
    long_full = long_prompt + response
    long_inputs = tokenizer2(long_full, return_tensors='pt')
    long_ids = long_inputs['input_ids']
    labels_long = long_ids.clone()
    labels_long[0, :len(tokenizer2.encode(long_prompt))] = -100
    outputs_long = model_demo(long_ids, labels=labels_long)
    loss_long = outputs_long.loss.item()

print(f"Loss WITHOUT masking:           {loss_no_mask:.4f}")
print(f"Loss WITH masking:              {loss_masked:.4f}")
print(f"Loss WITH masking (long prompt):{loss_long:.4f}")
print()
print("Without masking: loss is diluted by prompt tokens.")
print("The model wastes capacity learning to predict the instruction format.")
print()
print("With masking: loss reflects ONLY response quality — the signal you want.")
print("With very long prompts + masking: loss correctly focuses on the short response.")

## 4. Training Dynamics: Reading Loss Curves

In [ ]:
# Full training run with logging to study loss dynamics

from transformers import TrainerCallback

class LossLogger(TrainerCallback):
    """Captures step-level training and eval loss for analysis."""
    def __init__(self):
        self.train_losses = []
        self.eval_losses  = []
        self.steps        = []
        self.eval_steps   = []
        self.learning_rates = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.train_losses.append(logs['loss'])
            self.steps.append(state.global_step)
            if 'learning_rate' in logs:
                self.learning_rates.append(logs['learning_rate'])
        if logs and 'eval_loss' in logs:
            self.eval_losses.append(logs['eval_loss'])
            self.eval_steps.append(state.global_step)


# Training dataset: realistic instruction-following examples
train_examples = [
    {"instruction": "Explain backpropagation.",
     "response": "Backpropagation computes gradients by applying the chain rule backwards through a computational graph. Starting from the loss, it propagates error signals layer by layer, computing how much each weight contributed to the final error."},
    {"instruction": "What is dropout regularisation?",
     "response": "Dropout randomly deactivates neurons during training with probability p. This prevents co-adaptation — neurons cannot rely on specific others being present. At inference, all neurons are active but outputs are scaled by (1-p)."},
    {"instruction": "Explain the encoder-decoder architecture.",
     "response": "The encoder processes the input into a latent representation, capturing its meaning. The decoder generates output conditioned on this representation, attending to encoder states via cross-attention. Used in translation (Transformer), summarisation (BART), and image captioning."},
    {"instruction": "What is beam search?",
     "response": "Beam search maintains the top-k most probable partial sequences at each decoding step, where k is the beam width. Unlike greedy decoding (k=1), it explores multiple hypotheses simultaneously, typically finding higher-probability outputs at the cost of more computation."},
    {"instruction": "What causes the exploding gradient problem?",
     "response": "Exploding gradients occur when backpropagation multiplies many numbers greater than 1 together, causing exponential growth in gradient magnitude. Common in deep networks and RNNs. Solutions include gradient clipping (cap max gradient norm), careful weight initialisation, and residual connections."},
    {"instruction": "What is label smoothing?",
     "response": "Label smoothing replaces hard one-hot targets with soft distributions: the true class gets probability (1-ε) and all others share ε/(C-1), where C is the class count. This prevents overconfidence, improves calibration, and often improves generalisation."},
    {"instruction": "Explain positional encoding in transformers.",
     "response": "Positional encoding adds position information to token embeddings since transformers have no inherent sense of order. The original Transformer uses sinusoidal functions at different frequencies. Modern models often use Rotary Position Embeddings (RoPE) or Alibi, which extend better to longer sequences."},
    {"instruction": "What is the difference between BERT and GPT?",
     "response": "BERT is an encoder-only model trained with masked language modelling — it reads tokens bidirectionally and is optimised for understanding tasks. GPT is a decoder-only model trained autoregressively — it reads left-to-right and is optimised for generation. BERT excels at classification; GPT excels at text generation."},
] * 5  # Repeat for demo

eval_examples = train_examples[:4]

train_dataset = Dataset.from_list([{"text": format_instruction(e)} for e in train_examples])
eval_dataset  = Dataset.from_list([{"text": format_instruction(e)} for e in eval_examples])

# Model with LoRA
model_ft = AutoModelForCausalLM.from_pretrained("gpt2")
lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["c_attn"],
    task_type=TaskType.CAUSAL_LM
)
model_ft = get_peft_model(model_ft, lora_cfg)
model_ft.print_trainable_parameters()

logger = LossLogger()

training_args = TrainingArguments(
    output_dir="./instruct_output",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=10,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    max_seq_length=256,
    dataset_text_field="text",
    data_collator=DataCollatorForCompletionOnlyLM(
        response_template=RESPONSE_TEMPLATE,
        tokenizer=tokenizer
    ),
    callbacks=[logger],
)

print("Training...")
trainer.train()
print("Done!")

In [ ]:
# ── Analyse training dynamics ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Loss curves
ax = axes[0]
if logger.train_losses:
    ax.plot(logger.steps, logger.train_losses, color='#1f77b4', linewidth=2, label='Train loss')
if logger.eval_losses:
    ax.plot(logger.eval_steps, logger.eval_losses, 'o-', color='#d62728',
            linewidth=2, markersize=6, label='Eval loss')
ax.set_xlabel('Steps')
ax.set_ylabel('Loss')
ax.set_title('Training & Eval Loss')
ax.legend()
ax.grid(alpha=0.3)

# Learning rate schedule
ax = axes[1]
if logger.learning_rates:
    ax.plot(logger.steps[:len(logger.learning_rates)], logger.learning_rates,
            color='#2ca02c', linewidth=2)
ax.set_xlabel('Steps')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule with Warmup')
ax.grid(alpha=0.3)

# Reference: annotated healthy vs unhealthy curves
ax = axes[2]
x = np.linspace(0, 100, 200)
healthy  = 3.0 * np.exp(-x/30) + 0.3 + np.random.RandomState(1).randn(200) * 0.05
overfit  = np.array([3.0 * np.exp(-i/30) + 0.3 +
                     (0.0 if i < 50 else (i-50)*0.015) +
                     np.random.RandomState(42+int(i)).randn() * 0.03
                     for i in x])
unstable = 3.0 * np.exp(-x/50) + 0.3 + np.random.RandomState(7).randn(200) * 0.4
collapse = np.array([3.0 * np.exp(-i/30) + 0.3 if i < 40 else
                     3.0 * np.exp(-i/30) + 0.3 + (i-40)*0.08
                     for i in x])

ax.plot(x, healthy,  color='#2ca02c', linewidth=2, label='✅ Healthy')
ax.plot(x, overfit,  color='#ff7f0e', linewidth=2, label='⚠️  Overfitting', linestyle='--')
ax.plot(x, unstable, color='#9467bd', linewidth=1.5, label='⚠️  Unstable (high LR)', alpha=0.8)
ax.plot(x, collapse, color='#d62728', linewidth=2, label='❌ Loss collapse', linestyle='-.')
ax.set_xlabel('Steps')
ax.set_ylabel('Loss')
ax.set_title('Reference: Loss Curve Patterns')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_ylim(0, 5)

plt.suptitle('Instruction Tuning: Training Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/07_training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

print("Loss curve diagnostic guide:")
print("  ✅ Healthy:    smooth decrease, eval tracks train, converges gracefully")
print("  ⚠️  Overfitting: train keeps falling, eval starts rising — stop here!")
print("  ⚠️  Unstable:   high variance, spikes — reduce learning rate")
print("  ❌ Collapse:   loss starts rising after initial descent — catastrophic forgetting")

## 5. Multi-Turn Fine-Tuning

Single-turn (one instruction → one response) is simpler but real assistants need multi-turn conversation. The challenge: mask correctly across multiple turns.

In [ ]:
def format_multiturn_conversation(
    messages: List[Dict],
    tokenizer,
    train_on_all_responses: bool = True,
) -> Dict:
    """
    Format a multi-turn conversation for instruction tuning.

    Returns input_ids and labels with correct masking:
    - User turns: always masked (-100)
    - Assistant turns: active (contributes to loss)
    - System prompt: masked

    This is the format used by ChatML-style fine-tuning.
    """
    full_text = ""
    turn_boundaries = []  # (start, end, role) for each turn

    pos = 0
    for msg in messages:
        turn_text = f"### {msg['role'].upper()}:\n{msg['content']}\n\n"
        turn_start = pos
        full_text += turn_text
        pos += len(tokenizer.encode(turn_text, add_special_tokens=False))
        turn_boundaries.append((turn_start, pos, msg['role']))

    tokenized = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=512)
    input_ids = tokenized['input_ids'][0]
    labels    = input_ids.clone()

    # Mask all non-assistant tokens
    for start, end, role in turn_boundaries:
        if role != 'assistant':
            end_idx = min(end, len(labels))
            labels[start:end_idx] = -100

    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': tokenized['attention_mask'][0],
        'full_text': full_text,
    }


# Demo multi-turn conversation
conversation = [
    {"role": "system",    "content": "You are a helpful ML tutor."},
    {"role": "user",      "content": "What is a learning rate?"},
    {"role": "assistant", "content": "The learning rate controls the step size during gradient descent. A high rate converges fast but may overshoot; a low rate is stable but slow."},
    {"role": "user",      "content": "How do I choose a good learning rate?"},
    {"role": "assistant", "content": "Common strategies: (1) start with 1e-4 for transformers, (2) use a warmup then cosine decay, (3) try a learning rate range test to find the optimal range."},
]

result = format_multiturn_conversation(conversation, tokenizer)

total_tokens   = len(result['input_ids'])
active_tokens  = (result['labels'] != -100).sum().item()
masked_tokens  = total_tokens - active_tokens

print("Multi-turn conversation masking:")
print(f"  Total tokens:    {total_tokens}")
print(f"  Masked (user/system):    {masked_tokens}  ({100*masked_tokens/total_tokens:.0f}%)")
print(f"  Active (assistant):      {active_tokens}   ({100*active_tokens/total_tokens:.0f}%)")
print()

# Show which decoded tokens are active
print("Active response tokens (sample):")
for i, (tok_id, lab) in enumerate(zip(result['input_ids'][:60], result['labels'][:60])):
    if lab != -100:
        print(tokenizer.decode([tok_id]), end="")
print("...")

## 6. Evaluating Instruction Following

In [ ]:
def generate(
    model, tokenizer, instruction: str,
    max_new_tokens: int = 150,
    temperature: float = 0.7
) -> str:
    prompt = f"{INSTRUCTION_TEMPLATE}{instruction}\n\n{RESPONSE_TEMPLATE}"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Instruction-following evaluation suite
eval_suite = [
    {
        "category":     "Knowledge Recall",
        "instruction":  "What is gradient descent?",
        "keywords":     ["loss", "gradient", "parameter", "optimis"],
    },
    {
        "category":     "Format Following",
        "instruction":  "List three advantages of transformers over RNNs.",
        "keywords":     ["parallel", "attention", "long"],
        "format_check": lambda r: any(c in r for c in ['1.', '1)', '-', '•', '\n']),
    },
    {
        "category":     "Explanation",
        "instruction":  "Explain what overfitting means to a beginner.",
        "keywords":     ["train", "data", "new", "test"],
    },
    {
        "category":     "Out-of-domain",
        "instruction":  "Write a haiku about machine learning.",
        "keywords":     [],  # flexible
    },
]

print("Instruction-following evaluation:")
print("-" * 70)

for item in eval_suite:
    response = generate(model_ft, tokenizer, item['instruction'], max_new_tokens=100)

    # Keyword coverage
    if item['keywords']:
        hits = sum(1 for kw in item['keywords'] if kw.lower() in response.lower())
        keyword_score = hits / len(item['keywords'])
    else:
        keyword_score = None

    # Format check
    format_ok = item.get('format_check', lambda r: True)(response)

    print(f"[{item['category']}]")
    print(f"  Prompt:   {item['instruction']}")
    print(f"  Response: {response[:120]}{'...' if len(response)>120 else ''}")
    if keyword_score is not None:
        print(f"  Keyword coverage: {keyword_score:.0%} ({hits}/{len(item['keywords'])} keywords found)")
    print(f"  Format:   {'✅ OK' if format_ok else '⚠️  Check'}")
    print()

In [ ]:
# Compare base vs fine-tuned model side-by-side
base_model = AutoModelForCausalLM.from_pretrained("gpt2")

comparison_questions = [
    "What is dropout?",
    "Explain the difference between precision and recall.",
]

print("Base vs Fine-tuned Model Comparison")
print("=" * 70)
for q in comparison_questions:
    base_resp = generate(base_model, tokenizer, q, max_new_tokens=80)
    ft_resp   = generate(model_ft,   tokenizer, q, max_new_tokens=80)
    print(f"Question: {q}")
    print(f"  Base:  {base_resp[:120]}")
    print(f"  Tuned: {ft_resp[:120]}")
    print()

## 7. Common Failure Modes & Fixes

| Symptom | Root Cause | Fix |
|---------|-----------|-----|
| Model ignores format instructions | Prompt not masked — model learns to predict prompts | Use `DataCollatorForCompletionOnlyLM` |
| Model repeats itself | Too high temperature / too many training epochs | Lower temp, add repetition penalty, early stop |
| Catastrophic forgetting | LR too high, trained too long on narrow data | Lower LR, add replay data, use LoRA not full FT |
| Short / truncated responses | `max_new_tokens` too low, or EOS token in training data | Audit training data for spurious EOS; increase limit |
| Wrong language / mode at inference | Template mismatch between training and deployment | Hardcode template; always test with exact training format |
| Loss spikes mid-training | Single bad batch, gradient norm not clipped | Set `max_grad_norm=1.0`; audit dataset for outliers |
| Loss won't decrease | LR too low, LoRA rank too small, frozen wrong layers | Increase LR/rank; check `target_modules`; verify unfreezing |

## 8. Engineering Notes

```python
# Complete production checklist for instruction tuning

# 1. Data quality > data quantity
#    100 high-quality examples beat 10,000 noisy ones (Alpagasus paper)

# 2. Use DataCollatorForCompletionOnlyLM — never manually handle masking
collator = DataCollatorForCompletionOnlyLM(
    response_template="### Response:\n",
    tokenizer=tokenizer
)

# 3. Cosine LR schedule with warmup — always
lr_scheduler_type="cosine"
warmup_ratio=0.05  # ~5% of training for warmup

# 4. Gradient clipping for stability
max_grad_norm=1.0

# 5. Evaluate on a held-out set every N steps
eval_strategy="steps"
eval_steps=100

# 6. Use packing for efficiency with short examples
packing=True  # in SFTTrainer — combines short examples into one sequence

# 7. Always checkpoint — training can crash
save_strategy="steps"
save_steps=200
```

## 9. Exercises

1. **Template Sensitivity**: Take a fine-tuned model and run the same test questions with correct vs incorrect templates (e.g., Alpaca format vs ChatML). Measure the drop in response quality using an LLM judge.

2. **Mask Ablation**: Train two identical models — one with prompt masking, one without. Plot their training loss curves. After training, evaluate both on instruction-following quality. Which trains faster and scores better?

3. **Multi-turn Consistency**: Build a multi-turn dataset where conversations have 3–5 turns. Fine-tune and test whether the model can maintain context across the conversation using a reference-based evaluation.

4. **Data Efficiency Curve**: Train on 10, 50, 100, 500, 1000, and 5000 examples (same data, subset). Plot eval loss vs dataset size. Where are the returns diminishing?

5. **Catastrophic Forgetting**: Fine-tune a model on domain-specific data (e.g., medical QA), then evaluate it on a general benchmark (MMLU or TriviaQA). Measure the forgetting. Now repeat with LoRA instead of full fine-tuning. How much does LoRA protect against forgetting?

---

## 10. References

- [Wei et al. (2021) — Finetuned Language Models Are Zero-Shot Learners (FLAN)](https://arxiv.org/abs/2109.01652)
- [Wang et al. (2022) — Self-Instruct: Aligning Language Models with Self-Generated Instructions](https://arxiv.org/abs/2212.10560)
- [Chen et al. (2023) — Alpagasus: Training a Better Alpaca with Fewer Data](https://arxiv.org/abs/2307.08701)
- [Taori et al. (2023) — Stanford Alpaca: An Instruction-following LLaMA Model](https://github.com/tatsu-lab/stanford_alpaca)
- [TRL SFTTrainer Documentation](https://huggingface.co/docs/trl/sft_trainer)